In [61]:
import pandas as pd
import numpy as np
import json

In [ ]:
# Each record in the dataset stores a video every time it appears in the trending list -> one row

# IMPORT DEI DATI

In [ ]:
# List of countries
countries = ["CA", "DE", "FR", "GB", "IN", "JP", "KR", "MX", "RU", "US"]

# Create dictionaries where the keys are country codes
# and the values are the corresponding dataframes / JSON objects
csv_dfs = {}
json_data = {}

# Loop to automatically read all files
for country in countries:
    csv_path = "data/" + f"{country}videos.csv"
    json_path = "data/" + f"{country}_category_id.json"
    
    # Read CSV files
    df = pd.read_csv(csv_path, encoding="utf-8", encoding_errors="replace")  
    # YouTube files are not fully UTF-8 compliant
    # encoding_errors="replace" substitutes anomalous characters with a special symbol
    # in this case for Japanese/Chinese/Russian characters, etc.
    csv_dfs[country] = df  # dictionary with key = country, value = dataframe
    
    # Read JSON files (point 14)
    with open(json_path, encoding="utf-8") as f: 
        j = json.load(f) 
    json_data[country] = j

In [ ]:
# check that all countries are included
csv_dfs.keys(), json_data.keys()

(dict_keys(['CA', 'DE', 'FR', 'GB', 'IN', 'JP', 'KR', 'MX', 'RU', 'US']),
 dict_keys(['CA', 'DE', 'FR', 'GB', 'IN', 'JP', 'KR', 'MX', 'RU', 'US']))

In [65]:
csv_dfs["JP"].head() 

,video_id,trending_date,title,channel_title,category_id,publish_time,tags,views,likes,dislikes,comment_count,thumbnail_link,comments_disabled,ratings_disabled,video_error_or_removed,description
0,5ugKfHgsmYw,18.07.02,陸自ヘリ、垂直に落下＝路上の車が撮影,時事通信映像センター,25,2018-02-06T03:04:37.000Z,"事故|""佐賀""|""佐賀県""|""ヘリコプター""|""ヘリ""|""自衛隊""|""墜落""|""落下""|""現...",188085,591,189,0,https://i.ytimg.com/vi/5ugKfHgsmYw/default.jpg,True,False,False,佐賀県神埼市の民家に墜落した陸上自衛隊のＡＨ６４Ｄ戦闘ヘリコプターが垂直に落下する様子を、近...
1,ohObafdd34Y,18.07.02,イッテQ お祭り男宮川×手越 巨大ブランコ②,神谷えりな Kamiya Erina 2,1,2018-02-06T04:01:56.000Z,[none],90929,442,88,174,https://i.ytimg.com/vi/ohObafdd34Y/default.jpg,False,False,False,NaN
2,aBr2kKAHN6M,18.07.02,Live Views of Starman,SpaceX,28,2018-02-06T21:38:22.000Z,[none],6408303,165892,2331,3006,https://i.ytimg.com/vi/aBr2kKAHN6M/default.jpg,False,False,False,NaN
3,5wNnwChvmsQ,18.07.02,東京ディズニーリゾートの元キャストが暴露した秘密5選,アシタノワダイ,25,2018-02-06T06:08:49.000Z,アシタノワダイ,96255,1165,277,545,https://i.ytimg.com/vi/5wNnwChvmsQ/default.jpg,False,False,False,東京ディズニーリゾートの元キャストが暴露した秘密5選\n\nかたまりクリエイトさま\n【検証...
4,B7J47qFvdsk,18.07.02,榮倉奈々、衝撃の死んだふり！映画『家に帰ると妻が必ず死んだふりをしています。』特報,シネマトゥデイ,1,2018-02-06T02:30:00.000Z,[none],108408,1336,74,201,https://i.ytimg.com/vi/B7J47qFvdsk/default.jpg,False,False,False,家に帰ってきたサラリーマンのじゅん（安田顕）は、玄関で血を出して倒れている妻ちえ（榮倉奈々）...


# 1. Create a single dataframe with the concatenation of all input csv files,adding a column called country

In [ ]:
# to proceed with the first point, we check that
# column names are consistent across CSV files
for country, df in csv_dfs.items():  # items returns dictionary pairs (key = country, value = dataframe)
    print(country, " ", sorted(df.columns))

# they all seem consistent and without errors, we will check later if any issues arise

CA   ['category_id', 'channel_title', 'comment_count', 'comments_disabled', 'description', 'dislikes', 'likes', 'publish_time', 'ratings_disabled', 'tags', 'thumbnail_link', 'title', 'trending_date', 'video_error_or_removed', 'video_id', 'views']
DE   ['category_id', 'channel_title', 'comment_count', 'comments_disabled', 'description', 'dislikes', 'likes', 'publish_time', 'ratings_disabled', 'tags', 'thumbnail_link', 'title', 'trending_date', 'video_error_or_removed', 'video_id', 'views']
FR   ['category_id', 'channel_title', 'comment_count', 'comments_disabled', 'description', 'dislikes', 'likes', 'publish_time', 'ratings_disabled', 'tags', 'thumbnail_link', 'title', 'trending_date', 'video_error_or_removed', 'video_id', 'views']
GB   ['category_id', 'channel_title', 'comment_count', 'comments_disabled', 'description', 'dislikes', 'likes', 'publish_time', 'ratings_disabled', 'tags', 'thumbnail_link', 'title', 'trending_date', 'video_error_or_removed', 'video_id', 'views']
IN   ['categ

In [ ]:
# count the number of rows and columns:
for country, df in csv_dfs.items():
    print(country, df.shape)

# all have 16 columns. we will later verify that the new dataframe
# has 16 + 1 = 17 columns (after adding the country variable)

CA (40881, 16)
DE (40840, 16)
FR (40724, 16)
GB (38916, 16)
IN (37352, 16)
JP (20523, 16)
KR (34567, 16)
MX (40451, 16)
RU (40739, 16)
US (40949, 16)


In [ ]:
# now we create the new dataframe:

# create an empty list that we will use to store the dataframes
dfsss = []

# for each country and its corresponding dataframe in csv_dfs
for country, df in csv_dfs.items():
    
    # make a copy of the original dataframe
    # (this avoids accidentally modifying the one stored in the dictionary)
    temp = df.copy()
    
    # add a new column called "country"
    # and assign the country code as its value (e.g., "CA", "JP", "US", etc.)
    temp["country"] = country
    
    # append the modified dataframe to the list
    dfsss.append(temp)  # dfsss = [df_US, df_FR, df_JP, ...]

# then merge all dataframes in the list into one large dataframe
df2 = pd.concat(dfsss, ignore_index=True)  # ignore original indices and create a new one

# check
print("shape:", df2.shape)
df2.head()

# correct number of columns

shape: (375942, 17)


,video_id,trending_date,title,channel_title,category_id,publish_time,tags,views,likes,dislikes,comment_count,thumbnail_link,comments_disabled,ratings_disabled,video_error_or_removed,description,country
0,n1WpP7iowLc,17.14.11,Eminem - Walk On Water (Audio) ft. Beyoncé,EminemVEVO,10,2017-11-10T17:00:03.000Z,"Eminem|""Walk""|""On""|""Water""|""Aftermath/Shady/In...",17158579,787425,43420,125882,https://i.ytimg.com/vi/n1WpP7iowLc/default.jpg,False,False,False,Eminem's new track Walk on Water ft. Beyoncé i...,CA
1,0dBIkQ4Mz1M,17.14.11,PLUSH - Bad Unboxing Fan Mail,iDubbbzTV,23,2017-11-13T17:00:00.000Z,"plush|""bad unboxing""|""unboxing""|""fan mail""|""id...",1014651,127794,1688,13030,https://i.ytimg.com/vi/0dBIkQ4Mz1M/default.jpg,False,False,False,STill got a lot of packages. Probably will las...,CA
2,5qpjK5DgCt4,17.14.11,"Racist Superman | Rudy Mancuso, King Bach & Le...",Rudy Mancuso,23,2017-11-12T19:05:24.000Z,"racist superman|""rudy""|""mancuso""|""king""|""bach""...",3191434,146035,5339,8181,https://i.ytimg.com/vi/5qpjK5DgCt4/default.jpg,False,False,False,WATCH MY PREVIOUS VIDEO ▶ \n\nSUBSCRIBE ► http...,CA
3,d380meD0W0M,17.14.11,I Dare You: GOING BALD!?,nigahiga,24,2017-11-12T18:01:41.000Z,"ryan|""higa""|""higatv""|""nigahiga""|""i dare you""|""...",2095828,132239,1989,17518,https://i.ytimg.com/vi/d380meD0W0M/default.jpg,False,False,False,I know it's been a while since we did this sho...,CA
4,2Vv-BfVoq4g,17.14.11,Ed Sheeran - Perfect (Official Music Video),Ed Sheeran,10,2017-11-09T11:04:14.000Z,"edsheeran|""ed sheeran""|""acoustic""|""live""|""cove...",33523622,1634130,21082,85067,https://i.ytimg.com/vi/2Vv-BfVoq4g/default.jpg,False,False,False,🎧: https://ad.gt/yt-perfect\n💰: https://atlant...,CA


# 2. Extract all videos that have no tag.

In [ ]:
# check some information about the "tags" variable:
df2["tags"].apply(type).unique()

# all values are strings

array([<class 'str'>], dtype=object)

In [70]:
df2["tags"].head(10)

0    Eminem|"Walk"|"On"|"Water"|"Aftermath/Shady/In...
1    plush|"bad unboxing"|"unboxing"|"fan mail"|"id...
2    racist superman|"rudy"|"mancuso"|"king"|"bach"...
3    ryan|"higa"|"higatv"|"nigahiga"|"i dare you"|"...
4    edsheeran|"ed sheeran"|"acoustic"|"live"|"cove...
5    #DramaAlert|"Drama"|"Alert"|"DramaAlert"|"keem...
6    Funny Moments|"Montage video games"|"gaming"|"...
7                                      SHANtell martin
8    logan paul vlog|"logan paul"|"logan"|"paul"|"o...
9                 God|"Sheldon Cooper"|"Young Sheldon"
Name: tags, dtype: object

In [ ]:
# "tags" contains a string with tags separated by "|"

# let's try to find videos without tags by searching for empty strings
# or NaN values:

df2[df2["tags"].isna() | (df2["tags"] == "")]

# no rows are returned, meaning that videos without tags are not stored
# either as empty strings or as NaN values

,video_id,trending_date,title,channel_title,category_id,publish_time,tags,views,likes,dislikes,comment_count,thumbnail_link,comments_disabled,ratings_disabled,video_error_or_removed,description,country


In [ ]:
df2[df2["tags"].astype(str).str.strip().str.fullmatch(r"\|+")]
# select rows where the "tags" column contains ONLY
# one or more "|" characters and nothing else.

# NO match -> empty pipes do not represent missing tags

,video_id,trending_date,title,channel_title,category_id,publish_time,tags,views,likes,dislikes,comment_count,thumbnail_link,comments_disabled,ratings_disabled,video_error_or_removed,description,country


In [ ]:
# try to identify tag values that could indicate the absence
# of tags by finding those without the "|" separator
# (suspicious values with 0 "|" separators):
df2[df2["tags"].apply(lambda x: str(x).count("|") == 0)]

# it seems that the absence of tags is represented by the value "[none]".
# note that videos with only one tag (which do not contain "|")
# also appear in this result.

,video_id,trending_date,title,channel_title,category_id,publish_time,tags,views,likes,dislikes,comment_count,thumbnail_link,comments_disabled,ratings_disabled,video_error_or_removed,description,country
7,2kyS6SvSYSE,17.14.11,WE WANT TO TALK ABOUT OUR MARRIAGE,CaseyNeistat,22,2017-11-13T17:13:01.000Z,SHANtell martin,748374,57534,2967,15959,https://i.ytimg.com/vi/2kyS6SvSYSE/default.jpg,False,False,False,SHANTELL'S CHANNEL - https://www.youtube.com/s...,CA
41,JwboxqDylgg,17.14.11,Canada Soccer's Women's National Team v USA In...,Canada Soccer,17,2017-11-13T05:53:49.000Z,[none],36311,277,28,13,https://i.ytimg.com/vi/JwboxqDylgg/default.jpg,False,False,False,Canada Soccer's Women's National Team face riv...,CA
58,9B-q8h31Bpk,17.14.11,John Oliver Tackles Louis C.K. And Donald Trum...,TV Shows,22,2017-11-13T04:49:26.000Z,[none],106029,1270,101,181,https://i.ytimg.com/vi/9B-q8h31Bpk/default.jpg,False,False,False,"John Oliver on News, Politics ...",CA
78,1UE5Dq1rvUA,17.14.11,Taylor Swift Perform Ready For It - SNL,Ken Reactz,24,2017-11-12T05:18:02.000Z,[none],320964,8069,285,717,https://i.ytimg.com/vi/1UE5Dq1rvUA/default.jpg,False,False,False,Thanks for watching please subscribe and subsc...,CA
86,pmJQ4KwliX4,17.14.11,"LATEST Q POSTS: ROTHSCHILDS, HOUSE OF SAUD, lL...",James Munder,2,2017-11-12T21:25:40.000Z,[none],116820,1503,139,1066,https://i.ytimg.com/vi/pmJQ4KwliX4/default.jpg,False,False,False,https://pastebin.ca/3930472\n\nSupport My Chan...,CA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
375815,VGykknw9eCM,18.14.06,"Made Defiant: The Mixtape ft. Neymar Jr., Kane...",Beats by Dre,10,2018-06-07T07:01:06.000Z,[none],3577614,12318,1345,1414,https://i.ytimg.com/vi/VGykknw9eCM/default.jpg,False,False,False,"When your time comes, you have two choices. Yo...",US
375819,fAIX12F6958,18.14.06,Bumblebee (2018) - Official Teaser Trailer - P...,Paramount Pictures,1,2018-06-05T07:00:01.000Z,[none],19864779,228670,16420,46318,https://i.ytimg.com/vi/fAIX12F6958/default.jpg,False,False,False,Every adventure has a beginning. Watch the off...,US
375865,gS1DbvHHVH0,18.14.06,Going in to brain surgery,Simone Giertz,28,2018-05-30T14:22:13.000Z,[none],1635301,120791,1098,20711,https://i.ytimg.com/vi/gS1DbvHHVH0/default.jpg,False,False,False,We’ll post an update on Instagram and Twitter ...,US
375873,E4c7EE8_IX0,18.14.06,Weezer - Africa,weezer,10,2018-05-29T12:00:11.000Z,[none],4682557,115240,5069,9170,https://i.ytimg.com/vi/E4c7EE8_IX0/default.jpg,False,False,False,Stream + download the song: http://fanlink.to/...,US


In [ ]:
# identify "[none]" values.

# NOTE: this code does not find videos without tags, but OBSERVATIONS
# without tags. therefore, videos without tags (that were never updated
# with tags in later dates) APPEAR MULTIPLE TIMES.
no_tags = df2[df2["tags"].str.strip().str.lower() == "[none]"]

In [75]:
len(no_tags)

37698

In [76]:
no_tags.head()

,video_id,trending_date,title,channel_title,category_id,publish_time,tags,views,likes,dislikes,comment_count,thumbnail_link,comments_disabled,ratings_disabled,video_error_or_removed,description,country
41,JwboxqDylgg,17.14.11,Canada Soccer's Women's National Team v USA In...,Canada Soccer,17,2017-11-13T05:53:49.000Z,[none],36311,277,28,13,https://i.ytimg.com/vi/JwboxqDylgg/default.jpg,False,False,False,Canada Soccer's Women's National Team face riv...,CA
58,9B-q8h31Bpk,17.14.11,John Oliver Tackles Louis C.K. And Donald Trum...,TV Shows,22,2017-11-13T04:49:26.000Z,[none],106029,1270,101,181,https://i.ytimg.com/vi/9B-q8h31Bpk/default.jpg,False,False,False,"John Oliver on News, Politics ...",CA
78,1UE5Dq1rvUA,17.14.11,Taylor Swift Perform Ready For It - SNL,Ken Reactz,24,2017-11-12T05:18:02.000Z,[none],320964,8069,285,717,https://i.ytimg.com/vi/1UE5Dq1rvUA/default.jpg,False,False,False,Thanks for watching please subscribe and subsc...,CA
86,pmJQ4KwliX4,17.14.11,"LATEST Q POSTS: ROTHSCHILDS, HOUSE OF SAUD, lL...",James Munder,2,2017-11-12T21:25:40.000Z,[none],116820,1503,139,1066,https://i.ytimg.com/vi/pmJQ4KwliX4/default.jpg,False,False,False,https://pastebin.ca/3930472\n\nSupport My Chan...,CA
98,lHcXhBojpeQ,17.14.11,三屆TVB視帝，拋棄10年青梅竹馬髮妻，為娶小三還不惜與母絕交！,明星百曉生,22,2017-11-12T12:49:50.000Z,[none],88061,47,58,17,https://i.ytimg.com/vi/lHcXhBojpeQ/default.jpg,False,False,False,NaN,CA


In [ ]:
# since videos can appear multiple times in the dataset, and tags can be added or removed over time,
# we create a dataframe with unique videos by keeping one row per video (the most recent date).
df_unique = df2.sort_values("trending_date").drop_duplicates("video_id", keep="last")

In [ ]:
# extract unique videos that have "[none]" as tags
unique_no_tags = df_unique[df_unique["tags"].str.strip().str.lower() == "[none]"]

In [79]:
len(unique_no_tags)

23894

In [80]:
unique_no_tags.head()

,video_id,trending_date,title,channel_title,category_id,publish_time,tags,views,likes,dislikes,comment_count,thumbnail_link,comments_disabled,ratings_disabled,video_error_or_removed,description,country
222296,8YVTiNchnhU,17.01.12,"신의한수 생방송 11월 30일 / 트럼프, 김정은에 마지막 카드 꺼냈다!",신의한수,25,2017-11-30T10:01:44.000Z,[none],49486,2980,84,212,https://i.ytimg.com/vi/8YVTiNchnhU/default.jpg,False,False,False,(주)민초 커뮤니케이션 대표 신혜식입니다. 자발적 구독료는 정기 납부가 가능합니다....,KR
222294,i-2WX4dhKkk,17.01.12,한국은행 기준 금리 상승 충격파 가 증시 .부동산에 미칠 영향에 대하여....,도봉박홍기,25,2017-11-30T13:51:45.000Z,[none],27456,1675,46,49,https://i.ytimg.com/vi/i-2WX4dhKkk/default.jpg,False,False,False,우리 은행 (대동포럼) 박 홍기 후원계좌 1005-203-205878\n10만원...,KR
222305,uHBgnvwsfAs,17.01.12,[아라시] 매운 음식 먹다 이상증세를 보이는 4인(feat.리다 매운 거 맞지..?),SAKU 사쿠,22,2017-11-30T10:18:39.000Z,[none],25522,446,5,58,https://i.ytimg.com/vi/uHBgnvwsfAs/default.jpg,False,False,False,080825 아라시의 숙제군\n자막 : 아라시의 자막군\n\n쇼를 몰기 위해 필사적...,KR
222283,5SnLUOUhE0o,17.01.12,2017 MAMA WannaOne 강다니엘 리액션 모음,다니별,22,2017-11-29T23:37:26.000Z,[none],28881,962,5,105,https://i.ytimg.com/vi/5SnLUOUhE0o/default.jpg,False,False,False,인스타👉https://www.instagram.com/dani.star_love/\...,KR
257207,assNr4s0k5E,17.01.12,America vs Tigres 0-1 Gol y Resumen Semifinal ...,Brothers sports,17,2017-11-30T05:02:00.000Z,[none],353066,822,211,683,https://i.ytimg.com/vi/assNr4s0k5E/default.jpg,False,False,False,Descarga Onefootball gratis aquí: http://bit.d...,MX


# 3. For each channel, determine the total number of views


In [ ]:
# use df_unique, otherwise we would double count views of the same videos

# some checks:
df_unique["views"].dtype

# no issues with the views values; we can safely compute the sum

dtype('int64')

In [82]:
df_unique.groupby('channel_title')['views'].sum().sort_values(ascending=False)

channel_title
T-Series             537304749
NickyJamTV           384233475
Ed Sheeran           328330527
5-Minute Crafts      310334079
WWE                  288051034
                       ...    
NavylittleMonster          365
Videostendencias           302
No Comment TV              284
Sport Life                 163
Alexander Redking          153
Name: views, Length: 37575, dtype: int64

# 4. Save all rows with disabled comments and disabled ratings, or that have video_error_or_removed in a new dataframe called excluded, and remove those rows from the original dataframe.

In [ ]:
# in this case we use df2 because the task refers to rows, not unique videos
print(df2['comments_disabled'].dtype)
print(df2['ratings_disabled'].dtype)
print(df2['video_error_or_removed'].dtype)
print (df2['comments_disabled'].head(5))
# all three variables are boolean, so we can work with them without issues

bool
bool
bool
0    False
1    False
2    False
3    False
4    False
Name: comments_disabled, dtype: bool


In [ ]:
# select rows where:
# - comments and ratings are both disabled
#   OR
# - the video is marked as error or removed
excluded = df2[
    ((df2["comments_disabled"] == True) & (df2["ratings_disabled"] == True)) |
    (df2["video_error_or_removed"] == True)
]

# remove these rows from the original dataframe
df2 = df2.drop(excluded.index)

# (df2 now has gaps in the index)

In [86]:
len(excluded)
#2620 record

2620

# 5. Add a like_ratio column storing the ratio between the number of likes and of dislikes

In [ ]:
# first, check whether there are videos with 0 dislikes
# to avoid potential division by zero issues
df2[df2["dislikes"] == 0]

,video_id,trending_date,title,channel_title,category_id,publish_time,tags,views,likes,dislikes,comment_count,thumbnail_link,comments_disabled,ratings_disabled,video_error_or_removed,description,country
67,amEZKmJQ4Io,17.14.11,Drako - Watch Me Do It [Official Video],babygranderecords,10,2017-10-23T19:38:36.000Z,"Drako|""Watch Me Do It""|""watch me""|""migos""|""dap...",25887,0,0,6,https://i.ytimg.com/vi/amEZKmJQ4Io/default.jpg,False,True,False,PURCHASE / STREAM WATCH ME DO IT https://fanli...,CA
161,bsgocWaULIU,17.14.11,Troydon bent - Jam African,Troydon bent,10,2016-12-08T19:29:25.000Z,New Danchall music world reggae traditional so...,16921,187,0,5,https://i.ytimg.com/vi/bsgocWaULIU/default.jpg,False,False,False,Troydon bent performing Jam African produced b...,CA
235,wDEA3rpYHnI,17.15.11,Marie-Louise Arsenault réplique à Denise Bomba...,TV Classics,22,2017-11-13T01:26:37.000Z,Marie-Louise Arsenault qui réplique à Denise B...,15800,88,0,0,https://i.ytimg.com/vi/wDEA3rpYHnI/default.jpg,True,False,False,Moment favori à la télé québécoise: Marie-Loui...,CA
271,LfajgipdxmA,17.15.11,How does Yoonla work? Learning CPA,Anthonys Digital Lifestyle Dotcom,22,2017-11-14T10:57:38.000Z,"How does Yoonla work|""free work from home""|""le...",11810,1,0,0,https://i.ytimg.com/vi/LfajgipdxmA/default.jpg,False,False,False,How does Yoonla work? Learning CPA marketing t...,CA
313,8sdQ7dP9Drs,17.15.11,"Today's News: It's On - Get The Goods, Novembe...",Whistler Blackcomb,17,2017-11-14T18:18:21.000Z,[none],4851,31,0,0,https://i.ytimg.com/vi/8sdQ7dP9Drs/default.jpg,False,False,False,Today’s news: it’s on. Whistler Mountain will ...,CA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
358439,8WbxbqwdWog,18.13.03,Camera Goes on Japanese Sushi Conveyor Belt Sh...,TkyoSam,22,2018-03-06T16:18:30.000Z,"東京サム|""jvlog""|""jvlogger""|""tkyosam""|""tokyo""|""sam...",639517,0,0,1059,https://i.ytimg.com/vi/8WbxbqwdWog/default.jpg,False,True,False,I will call this video A Camera's Journey Thro...,US
360544,zkrq7Kpd1so,18.24.03,ULTRA LIVE presents Ultra Music Festival 2018 ...,UMF TV,10,2018-03-24T07:55:21.000Z,"ultra music festival|""umf""|""mix""|""relive""|""aft...",1648886,0,0,548,https://i.ytimg.com/vi/zkrq7Kpd1so/default.jpg,False,True,False,Tune in here for the Ultra Music Festival 2018...,US
363543,9TUBf6l7FBg,18.14.04,Coachella 2018 LIVE Channel 1,Coachella,10,2018-04-05T06:48:28.000Z,[none],3598220,0,0,0,https://i.ytimg.com/vi/9TUBf6l7FBg/default_liv...,False,True,False,For more cameras and VR180 immersive experienc...,US
363744,9TUBf6l7FBg,18.15.04,Coachella 2018 LIVE Channel 1,Coachella,10,2018-04-05T06:48:28.000Z,[none],11137071,0,0,2,https://i.ytimg.com/vi/9TUBf6l7FBg/default_liv...,False,True,False,For more cameras and VR180 immersive experienc...,US


In [ ]:
df2[df2["likes"] == 0]

# since there are some, we need to consider that the like_ratio defined as likes/dislikes
# will be NaN when a video has both 0 likes and 0 dislikes,
# and inf when it has a positive number of likes and 0 dislikes,
# and 0 when it has 0 likes for any positive number of dislikes.
# we should use an alternative formulation to avoid losing information.

,video_id,trending_date,title,channel_title,category_id,publish_time,tags,views,likes,dislikes,comment_count,thumbnail_link,comments_disabled,ratings_disabled,video_error_or_removed,description,country
67,amEZKmJQ4Io,17.14.11,Drako - Watch Me Do It [Official Video],babygranderecords,10,2017-10-23T19:38:36.000Z,"Drako|""Watch Me Do It""|""watch me""|""migos""|""dap...",25887,0,0,6,https://i.ytimg.com/vi/amEZKmJQ4Io/default.jpg,False,True,False,PURCHASE / STREAM WATCH ME DO IT https://fanli...,CA
385,ClwBTkLiivk,17.15.11,Week 4 Challenge: Check out “My Email Leads”,CREA | ACI,29,2017-11-13T13:46:37.000Z,CREA,1898,0,0,0,https://i.ytimg.com/vi/ClwBTkLiivk/default.jpg,True,False,False,www.realtor.ca/2mins,CA
489,HnCU20Cu0fs,17.16.11,ORIGINAL: Dashcam Norway - Semi truck narrowly...,Transferd AS,29,2017-11-15T07:37:18.000Z,"dashcam|""accident""|""traffic""|""safety""|""volvo""|...",2510664,0,0,1641,https://i.ytimg.com/vi/HnCU20Cu0fs/default.jpg,False,True,False,Copyright Transferd AS - a part of the Firda B...,CA
558,5khTN_EnAlY,17.16.11,（画在脸上的SEPHORA HAUL！) 红色秋冬新品妆容 | 小眼睛怎么hold住大热的...,Rainie Tian,26,2017-11-15T21:17:54.000Z,"sephora新品妆容|""2017化妆品新品""|""sephorahaul""|""跟我边聊边画""...",40407,0,0,319,https://i.ytimg.com/vi/5khTN_EnAlY/default.jpg,False,True,False,ALOHA 小伙伴们! 这次是一个不一样的sephora haul 哈哈 \n今天画了一个妖...,CA
579,0ayARJdf7I4,17.16.11,018 Algebra Lineal 18-10-2011,udearroba,27,2017-11-15T18:20:06.000Z,[none],1141,0,0,0,https://i.ytimg.com/vi/0ayARJdf7I4/default.jpg,False,False,False,Base ortogonal,CA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
358439,8WbxbqwdWog,18.13.03,Camera Goes on Japanese Sushi Conveyor Belt Sh...,TkyoSam,22,2018-03-06T16:18:30.000Z,"東京サム|""jvlog""|""jvlogger""|""tkyosam""|""tokyo""|""sam...",639517,0,0,1059,https://i.ytimg.com/vi/8WbxbqwdWog/default.jpg,False,True,False,I will call this video A Camera's Journey Thro...,US
360544,zkrq7Kpd1so,18.24.03,ULTRA LIVE presents Ultra Music Festival 2018 ...,UMF TV,10,2018-03-24T07:55:21.000Z,"ultra music festival|""umf""|""mix""|""relive""|""aft...",1648886,0,0,548,https://i.ytimg.com/vi/zkrq7Kpd1so/default.jpg,False,True,False,Tune in here for the Ultra Music Festival 2018...,US
363543,9TUBf6l7FBg,18.14.04,Coachella 2018 LIVE Channel 1,Coachella,10,2018-04-05T06:48:28.000Z,[none],3598220,0,0,0,https://i.ytimg.com/vi/9TUBf6l7FBg/default_liv...,False,True,False,For more cameras and VR180 immersive experienc...,US
363744,9TUBf6l7FBg,18.15.04,Coachella 2018 LIVE Channel 1,Coachella,10,2018-04-05T06:48:28.000Z,[none],11137071,0,0,2,https://i.ytimg.com/vi/9TUBf6l7FBg/default_liv...,False,True,False,For more cameras and VR180 immersive experienc...,US


In [ ]:
df2["like_ratio"] = (df2["likes"] + 1) / (df2["dislikes"] + 1)

# add a small pseudo-count to avoid undefined ratios and preserve information,
# especially when likes are 0 and dislikes are large.
df2.head()

,video_id,trending_date,title,channel_title,category_id,publish_time,tags,views,likes,dislikes,comment_count,thumbnail_link,comments_disabled,ratings_disabled,video_error_or_removed,description,country,like_ratio
0,n1WpP7iowLc,17.14.11,Eminem - Walk On Water (Audio) ft. Beyoncé,EminemVEVO,10,2017-11-10T17:00:03.000Z,"Eminem|""Walk""|""On""|""Water""|""Aftermath/Shady/In...",17158579,787425,43420,125882,https://i.ytimg.com/vi/n1WpP7iowLc/default.jpg,False,False,False,Eminem's new track Walk on Water ft. Beyoncé i...,CA,18.134681
1,0dBIkQ4Mz1M,17.14.11,PLUSH - Bad Unboxing Fan Mail,iDubbbzTV,23,2017-11-13T17:00:00.000Z,"plush|""bad unboxing""|""unboxing""|""fan mail""|""id...",1014651,127794,1688,13030,https://i.ytimg.com/vi/0dBIkQ4Mz1M/default.jpg,False,False,False,STill got a lot of packages. Probably will las...,CA,75.663114
2,5qpjK5DgCt4,17.14.11,"Racist Superman | Rudy Mancuso, King Bach & Le...",Rudy Mancuso,23,2017-11-12T19:05:24.000Z,"racist superman|""rudy""|""mancuso""|""king""|""bach""...",3191434,146035,5339,8181,https://i.ytimg.com/vi/5qpjK5DgCt4/default.jpg,False,False,False,WATCH MY PREVIOUS VIDEO ▶ \n\nSUBSCRIBE ► http...,CA,27.347566
3,d380meD0W0M,17.14.11,I Dare You: GOING BALD!?,nigahiga,24,2017-11-12T18:01:41.000Z,"ryan|""higa""|""higatv""|""nigahiga""|""i dare you""|""...",2095828,132239,1989,17518,https://i.ytimg.com/vi/d380meD0W0M/default.jpg,False,False,False,I know it's been a while since we did this sho...,CA,66.452261
4,2Vv-BfVoq4g,17.14.11,Ed Sheeran - Perfect (Official Music Video),Ed Sheeran,10,2017-11-09T11:04:14.000Z,"edsheeran|""ed sheeran""|""acoustic""|""live""|""cove...",33523622,1634130,21082,85067,https://i.ytimg.com/vi/2Vv-BfVoq4g/default.jpg,False,False,False,🎧: https://ad.gt/yt-perfect\n💰: https://atlant...,CA,77.509415


# 6. Cluster the publish time into 10-minute intervals (e.g. from 02:20 to 02:30)

In [ ]:
# perform some preliminary checks
df2['publish_time'].dtype
# the column is treated as a string (object)

dtype('O')

In [ ]:
#check structure
df2['publish_time'].head()

0    2017-11-10T17:00:03.000Z
1    2017-11-13T17:00:00.000Z
2    2017-11-12T19:05:24.000Z
3    2017-11-12T18:01:41.000Z
4    2017-11-09T11:04:14.000Z
Name: publish_time, dtype: object

In [ ]:
df2['publish_time'].isna().sum() #no missing values

0

In [ ]:
df2['publish_time_dt'] = pd.to_datetime(df2['publish_time'], errors='coerce')

# create a new column with the publication time converted to datetime format.
df2[['publish_time_dt']].head()

,publish_time_dt
0,2017-11-10 17:00:03+00:00
1,2017-11-13 17:00:00+00:00
2,2017-11-12 19:05:24+00:00
3,2017-11-12 18:01:41+00:00
4,2017-11-09 11:04:14+00:00


In [ ]:
df2[df2["publish_time_dt"].isna()]
#no missing or NaT

,video_id,trending_date,title,channel_title,category_id,publish_time,tags,views,likes,dislikes,comment_count,thumbnail_link,comments_disabled,ratings_disabled,video_error_or_removed,description,country,like_ratio,publish_time_dt


In [ ]:
# define the start and end of 10-minute intervals
start = df2["publish_time_dt"].dt.floor("10T")
# round each publication time down to the nearest 10-minute block

end = start + pd.Timedelta(minutes=10)
# add 10 minutes to define the upper bound of the interval

# label the interval using only hours and minutes
df2["publish_time_interval"] = (
    start.dt.strftime("%H:%M") + " - " + end.dt.strftime("%H:%M")
)

In [97]:
#check 
df2.head()

,video_id,trending_date,title,channel_title,category_id,publish_time,tags,views,likes,dislikes,comment_count,thumbnail_link,comments_disabled,ratings_disabled,video_error_or_removed,description,country,like_ratio,publish_time_dt,publish_time_interval
0,n1WpP7iowLc,17.14.11,Eminem - Walk On Water (Audio) ft. Beyoncé,EminemVEVO,10,2017-11-10T17:00:03.000Z,"Eminem|""Walk""|""On""|""Water""|""Aftermath/Shady/In...",17158579,787425,43420,125882,https://i.ytimg.com/vi/n1WpP7iowLc/default.jpg,False,False,False,Eminem's new track Walk on Water ft. Beyoncé i...,CA,18.134681,2017-11-10 17:00:03+00:00,17:00 - 17:10
1,0dBIkQ4Mz1M,17.14.11,PLUSH - Bad Unboxing Fan Mail,iDubbbzTV,23,2017-11-13T17:00:00.000Z,"plush|""bad unboxing""|""unboxing""|""fan mail""|""id...",1014651,127794,1688,13030,https://i.ytimg.com/vi/0dBIkQ4Mz1M/default.jpg,False,False,False,STill got a lot of packages. Probably will las...,CA,75.663114,2017-11-13 17:00:00+00:00,17:00 - 17:10
2,5qpjK5DgCt4,17.14.11,"Racist Superman | Rudy Mancuso, King Bach & Le...",Rudy Mancuso,23,2017-11-12T19:05:24.000Z,"racist superman|""rudy""|""mancuso""|""king""|""bach""...",3191434,146035,5339,8181,https://i.ytimg.com/vi/5qpjK5DgCt4/default.jpg,False,False,False,WATCH MY PREVIOUS VIDEO ▶ \n\nSUBSCRIBE ► http...,CA,27.347566,2017-11-12 19:05:24+00:00,19:00 - 19:10
3,d380meD0W0M,17.14.11,I Dare You: GOING BALD!?,nigahiga,24,2017-11-12T18:01:41.000Z,"ryan|""higa""|""higatv""|""nigahiga""|""i dare you""|""...",2095828,132239,1989,17518,https://i.ytimg.com/vi/d380meD0W0M/default.jpg,False,False,False,I know it's been a while since we did this sho...,CA,66.452261,2017-11-12 18:01:41+00:00,18:00 - 18:10
4,2Vv-BfVoq4g,17.14.11,Ed Sheeran - Perfect (Official Music Video),Ed Sheeran,10,2017-11-09T11:04:14.000Z,"edsheeran|""ed sheeran""|""acoustic""|""live""|""cove...",33523622,1634130,21082,85067,https://i.ytimg.com/vi/2Vv-BfVoq4g/default.jpg,False,False,False,🎧: https://ad.gt/yt-perfect\n💰: https://atlant...,CA,77.509415,2017-11-09 11:04:14+00:00,11:00 - 11:10


# 7. For each interval, determine the number of videos, average number of likes and of dislikes.

In [ ]:

interval_stats = df2.groupby("publish_time_interval").agg(
    num_videos=("publish_time_dt", "count"),
    avg_likes=("likes", "mean"),
    avg_dislikes=("dislikes", "mean")
).reset_index()

# group by publish_time_interval
# then compute count and mean for the selected variables

interval_stats.head()

# note: avg_likes and avg_dislikes count the same video multiple times
# if it appeared in trending on different dates

,publish_time_interval,num_videos,avg_likes,avg_dislikes
0,00:00 - 00:10,2897,61288.115637,3808.149465
1,00:10 - 00:20,1509,22748.138502,1449.836315
2,00:20 - 00:30,1241,21378.280419,1072.344883
3,00:30 - 00:40,1614,36853.560719,955.890954
4,00:40 - 00:50,1269,42198.623325,1909.301812


# Creazione della colonna like_ratio in df_unique

In [ ]:
# before proceeding with points 8 and 9, create this variable for point 10
# so that the ratio is already computed in df_unique before exploding
df_unique["like_ratio"] = (df_unique["likes"] + 1) / (df_unique["dislikes"] + 1)

# 8. For each tag, determine the number of videos. Notice that tags contains a string with several tags.
# AND 
# 9. Find the tags with the largest number of videos

In [ ]:
# in this case, a video that trended many times would be counted with multiple tag occurrences,
# so it makes more sense to work with unique videos in df_unique.

# there are cases with double pipes and strings ending with a pipe,
# which would generate empty tags.
# we take this into account when performing point 8.

In [ ]:
# create a dataframe excluding "[none]" tags (we can’t reuse the previous selection because it was based on df2)
df_tags = df_unique[df_unique["tags"] != "[none]"].copy()

# convert the tag string into a list of tags
df_tags["tags_list"] = df_tags["tags"].apply(lambda x: x.split("|"))

# explode: one row per tag
tags_exploded = df_tags.explode("tags_list")

# remove any residual empty tags (e.g., due to double pipes or trailing pipes)
tags_exploded = tags_exploded[tags_exploded["tags_list"].str.strip() != ""]

# tag counts (sorted for point 9)
tag_counts = (
    tags_exploded
    .groupby("tags_list")
    .size()
    .sort_values(ascending=False)
)

tag_counts.head(20)

tags_list
"2018"          5564
"funny"         4093
"comedy"        3041
"news"          2816
"2017"          2572
"show"          2034
"video"         2014
"television"    1713
"tv"            1696
"music"         1465
"новости"       1394
"юмор"          1375
"политика"      1372
"humor"         1329
"humour"        1328
"vlog"          1320
"TV"            1312
"review"        1290
"live"          1261
"обзор"         1256
dtype: int64

# 10. For each (tag, country) pair, compute average ratio likes/dislikes

In [ ]:
# we want the average ratio of all videos that have that tag within each country.
# we don’t want to count the same video’s ratio multiple times -> use df_unique
tag_country_mean_ratio = (
    tags_exploded
    .groupby(["tags_list", "country"])["like_ratio"]
    .mean()
    .reset_index(name="mean_like_ratio")
)
tag_country_mean_ratio.head(20)

,tags_list,country,mean_like_ratio
0,"#Freeticket""",IN,2.653482
1,#GST,IN,2.415596
2,#Jaisimha,IN,2.653482
3,#JanaSenaParty,IN,17.075812
4,"#MahaaNews""",IN,8.295547
5,#PawanKalyan #AlluArjun,IN,9.383399
6,"#PrimeTimeWithMurthy""",IN,6.252155
7,#RamGopalVarma,IN,2.415596
8,"#SUSANNE""",DE,77.903614
9,#VMA 20,FR,9.521739


In [ ]:
# for example, retrieve all like_ratio values for the tag '2018' in each country

# use a filter with query
tag_country_mean_ratio.query("tags_list == '2018'")

,tags_list,country,mean_like_ratio
934503,2018,CA,69.254150
934504,2018,DE,22.156781
934505,2018,FR,6.378641
934506,2018,GB,20.728850
934507,2018,IN,32.712891
934508,2018,JP,9.138208
934509,2018,KR,17.060996
934510,2018,MX,16.088118
934511,2018,RU,16.158595
934512,2018,US,9.378443


# 11. For each (trending_date, country) pair, the video with the largest number of views

In [ ]:
# use df_unique to minimize redundant operations, since a video's views can only increase
# and we consider the most recent record

video_max_views = df_unique.loc[
    df_unique.groupby(["trending_date", "country"])["views"].idxmax()
].reset_index(drop=True)

video_max_views.head(20)


,video_id,trending_date,title,channel_title,category_id,publish_time,tags,views,likes,dislikes,comment_count,thumbnail_link,comments_disabled,ratings_disabled,video_error_or_removed,description,country,like_ratio
0,08XqNxVgfMU,17.01.12,DONALD TRUMP SECRET VIDEO,PewDiePie,23,2017-11-27T19:45:53.000Z,"pewdiepie|""you laugh you lose""|""challenge""",2716883,236848,3681,16124,https://i.ytimg.com/vi/08XqNxVgfMU/default.jpg,False,False,False,Join my giveaway for a BEAST Origin computer! ...,CA,64.326181
1,UvjdPQs2kwU,17.01.12,Söz | 24.Bölüm - Fragman 1,Söz Dizi,1,2017-11-29T17:38:37.000Z,"söz|""soz""|""tims&b""|""tolga sarıtaş""|""aybüke pus...",1410983,25645,366,1808,https://i.ytimg.com/vi/UvjdPQs2kwU/default.jpg,False,False,False,'SÖZ' Yeni Bölümüyle Pazartesi 20:00'da Star T...,DE,69.880109
2,SnJYphO6NtU,17.01.12,"Making of Rani za3efane, Anes Tina , كواليس را...",Anes Tina,23,2017-11-30T17:15:50.000Z,Anes tina making of rani za3fan,612114,33023,780,4210,https://i.ytimg.com/vi/SnJYphO6NtU/default.jpg,False,False,False,Abonnez vous a ma chaîne youtube,FR,42.284251
3,jrfsschAssI,17.01.12,Liam Payne Answers Ellen’s Burning Questions,TheEllenShow,24,2017-11-30T18:00:03.000Z,"Liam Payne|""liam""|""payne""|""Ellen""|""Ellen's bur...",167391,7780,116,290,https://i.ytimg.com/vi/jrfsschAssI/default.jpg,False,False,False,The Bedroom Floor singer takes on Ellen’s most...,GB,66.504274
4,1vmhLprZYBg,17.01.12,Arsenal vs Huddersfield 5-0 All Goals & Highli...,Wrsh98,17,2017-11-29T22:32:58.000Z,Arsenal vs Huddersfield 5-0 All Goals & Highli...,831373,2157,505,225,https://i.ytimg.com/vi/1vmhLprZYBg/default.jpg,False,False,False,Hit LIKE and SUBSCRIBE\nThank you for watching...,IN,4.264822
5,o14K6dQwqTo,17.01.12,엄태웅 부인 윤혜진 이야기 너무나 안타깝다.,조회수 3.186.283회,24,2017-11-29T12:07:56.000Z,"엄태웅|""UCtIqlGteWia0_HgUxsm6oZA""",246211,265,145,31,https://i.ytimg.com/vi/o14K6dQwqTo/default.jpg,False,False,False,뉴스공장 : 엄태웅 부인 윤혜진 이야기 너무나 안타깝다. \n\n++++++++++...,KR,1.821918
6,SKKzJzpM3hI,17.01.12,"SIN MIEDO esperan el SORTEO MESSI, CR7 y NEYMA...",Cracks,17,2017-11-30T17:29:17.000Z,"Cracks|""UCL""|""Manu Bravo""|""Fútbol""|""Futebol""|""...",817123,41544,530,1163,https://i.ytimg.com/vi/SKKzJzpM3hI/default.jpg,False,False,False,Guardiola: “Mendy está LOCO”. Así se jugará el...,MX,78.239171
7,GtBlEIoE7Do,17.01.12,САМЫЙ НЕЛОВКИЙ МОМЕНТ!!! / MTV EMA 2017 #ЛОНДОН,TheKateClapp,10,2017-11-30T10:59:44.000Z,"катя|""клэп""|""кейт""|""kate""|""clapp""|""mtv""|""emine...",584166,49818,822,2147,https://i.ytimg.com/vi/GtBlEIoE7Do/default.jpg,False,False,False,Поездка в Лондон на MTV EMA 2017 случилась!!! ...,RU,60.533414
8,we2XODhx-CM,17.01.12,Chelsea Handler Questions Whether She's Cut Ou...,TheEllenShow,24,2017-11-30T14:00:07.000Z,"Chelsea|""Handler""|""Chelsea Handler""|""Kate Midd...",264975,5214,217,308,https://i.ytimg.com/vi/we2XODhx-CM/default.jpg,False,False,False,The always hilarious Chelsea Handler dished to...,US,23.922018
9,Pic_ayMVI-I,17.02.12,《爸爸去哪儿5》第12期完整版20171130: 大结局（上）鲫鱼兄弟携阿拉蕾回归 爸爸团开...,湖南卫视芒果TV官方频道 China HunanTV Official Channel,24,2017-11-30T16:00:01.000Z,"爸爸去哪儿|""爸爸去哪兒""|""爸爸去哪儿第五季""|""爸爸去哪儿5""|""爸爸去哪儿2017""|...",1038976,4297,341,2196,https://i.ytimg.com/vi/Pic_ayMVI-I/default.jpg,False,False,False,【欢迎订阅湖南卫视官方频道 Subscribe to Hunan TV YouTube Ch...,CA,12.567251


# 12. Divide trending_date into three columns: year, month, day

In [107]:
df2['trending_date'].head(5)

0    17.14.11
1    17.14.11
2    17.14.11
3    17.14.11
4    17.14.11
Name: trending_date, dtype: object

In [ ]:
# first, check the format of trending_date and note that
# it is not in a standard format, but YY.DD.MM (not YYYY-MM-DD)

# we need to explicitly specify the correct format when converting it

df2["trending_date"] = pd.to_datetime(df2["trending_date"], format="%y.%d.%m")

df2["year"] = df2["trending_date"].dt.year
df2["month"] = df2["trending_date"].dt.month
df2["day"] = df2["trending_date"].dt.day

In [ ]:
#check:
df2.head()

,video_id,trending_date,title,channel_title,category_id,publish_time,tags,views,likes,dislikes,...,ratings_disabled,video_error_or_removed,description,country,like_ratio,publish_time_dt,publish_time_interval,year,month,day
0,n1WpP7iowLc,2017-11-14,Eminem - Walk On Water (Audio) ft. Beyoncé,EminemVEVO,10,2017-11-10T17:00:03.000Z,"Eminem|""Walk""|""On""|""Water""|""Aftermath/Shady/In...",17158579,787425,43420,...,False,False,Eminem's new track Walk on Water ft. Beyoncé i...,CA,18.134681,2017-11-10 17:00:03+00:00,17:00 - 17:10,2017,11,14
1,0dBIkQ4Mz1M,2017-11-14,PLUSH - Bad Unboxing Fan Mail,iDubbbzTV,23,2017-11-13T17:00:00.000Z,"plush|""bad unboxing""|""unboxing""|""fan mail""|""id...",1014651,127794,1688,...,False,False,STill got a lot of packages. Probably will las...,CA,75.663114,2017-11-13 17:00:00+00:00,17:00 - 17:10,2017,11,14
2,5qpjK5DgCt4,2017-11-14,"Racist Superman | Rudy Mancuso, King Bach & Le...",Rudy Mancuso,23,2017-11-12T19:05:24.000Z,"racist superman|""rudy""|""mancuso""|""king""|""bach""...",3191434,146035,5339,...,False,False,WATCH MY PREVIOUS VIDEO ▶ \n\nSUBSCRIBE ► http...,CA,27.347566,2017-11-12 19:05:24+00:00,19:00 - 19:10,2017,11,14
3,d380meD0W0M,2017-11-14,I Dare You: GOING BALD!?,nigahiga,24,2017-11-12T18:01:41.000Z,"ryan|""higa""|""higatv""|""nigahiga""|""i dare you""|""...",2095828,132239,1989,...,False,False,I know it's been a while since we did this sho...,CA,66.452261,2017-11-12 18:01:41+00:00,18:00 - 18:10,2017,11,14
4,2Vv-BfVoq4g,2017-11-14,Ed Sheeran - Perfect (Official Music Video),Ed Sheeran,10,2017-11-09T11:04:14.000Z,"edsheeran|""ed sheeran""|""acoustic""|""live""|""cove...",33523622,1634130,21082,...,False,False,🎧: https://ad.gt/yt-perfect\n💰: https://atlant...,CA,77.509415,2017-11-09 11:04:14+00:00,11:00 - 11:10,2017,11,14


# 13. For each (month, country) pair, the video with the largest number of views

In [ ]:
# same as point 11, but performed on df2 since the "month" column is already available

video_max_views_month = (
    df2.loc[df2.groupby(["month", "country"])["views"].idxmax()]
    .reset_index(drop=True)
    .sort_values(["month", "country"])
)

video_max_views_month.head(20)


,video_id,trending_date,title,channel_title,category_id,publish_time,tags,views,likes,dislikes,...,ratings_disabled,video_error_or_removed,description,country,like_ratio,publish_time_dt,publish_time_interval,year,month,day
0,LsoLEjrDogU,2018-01-09,Bruno Mars - Finesse (Remix) [Feat. Cardi B] [...,Bruno Mars,10,2018-01-04T04:49:43.000Z,"Bruno Mars|""Finesse""|""Cardi B""|""Finesse Remix""...",43067983,1717177,61567,...,False,False,Finesse (Remix) Feat. Cardi B Available Now: h...,CA,27.890755,2018-01-04 04:49:43+00:00,04:40 - 04:50,2018,1,9
1,LsoLEjrDogU,2018-01-08,Bruno Mars - Finesse (Remix) [Feat. Cardi B] [...,Bruno Mars,10,2018-01-04T04:49:43.000Z,"Bruno Mars|""Finesse""|""Cardi B""|""Finesse Remix""...",37728802,1629946,56305,...,False,False,Finesse (Remix) Feat. Cardi B Available Now: h...,DE,28.948016,2018-01-04 04:49:43+00:00,04:40 - 04:50,2018,1,8
2,LsoLEjrDogU,2018-01-08,Bruno Mars - Finesse (Remix) [Feat. Cardi B] [...,Bruno Mars,10,2018-01-04T04:49:43.000Z,"Bruno Mars""|""Finesse""|""Cardi B""|""Finesse Remix...",37728802,1629948,56305,...,False,False,Finesse (Remix) Feat. Cardi B Available Now: h...,FR,28.948052,2018-01-04 04:49:43+00:00,04:40 - 04:50,2018,1,8
3,LsoLEjrDogU,2018-01-18,Bruno Mars - Finesse (Remix) [Feat. Cardi B] [...,Bruno Mars,10,2018-01-04T04:49:43.000Z,"Bruno Mars|""Finesse""|""Cardi B""|""Finesse Remix""...",90598955,2248693,93089,...,False,False,Finesse (Remix) Feat. Cardi B Available Now: h...,GB,24.156128,2018-01-04 04:49:43+00:00,04:40 - 04:50,2018,1,18
4,dfnCAmr569k,2018-01-18,"Taylor Swift - End Game ft. Ed Sheeran, Future",TaylorSwiftVEVO,10,2018-01-12T05:00:01.000Z,"Taylor|""Swift""|""End""|""Game""|""Big""|""Machine""|""Pop""",42019590,1804377,100033,...,False,False,Music video by Taylor Swift performing End Gam...,IN,18.037647,2018-01-12 05:00:01+00:00,05:00 - 05:10,2018,1,18
5,LsoLEjrDogU,2018-01-08,Bruno Mars - Finesse (Remix) [Feat. Cardi B] [...,Bruno Mars,10,2018-01-04T04:49:43.000Z,"Bruno Mars|""Finesse""|""Cardi B""|""Finesse Remix""...",37728802,1629948,56304,...,False,False,Finesse (Remix) Feat. Cardi B Available Now: h...,KR,28.948566,2018-01-04 04:49:43+00:00,04:40 - 04:50,2018,1,8
6,LsoLEjrDogU,2018-01-07,Bruno Mars - Finesse (Remix) [Feat. Cardi B] [...,Bruno Mars,10,2018-01-04T04:49:43.000Z,"Bruno Mars|""Finesse""|""Cardi B""|""Finesse Remix""...",31680160,1510636,49497,...,False,False,Finesse (Remix) Feat. Cardi B Available Now: h...,MX,30.519152,2018-01-04 04:49:43+00:00,04:40 - 04:50,2018,1,7
7,dfnCAmr569k,2018-01-14,"Taylor Swift - End Game ft. Ed Sheeran, Future",TaylorSwiftVEVO,10,2018-01-12T05:00:01.000Z,"Taylor|""Swift""|""End""|""Game""|""Big""|""Machine""|""Pop""",23198594,1404648,72534,...,False,False,Music video by Taylor Swift performing End Gam...,RU,19.365120,2018-01-12 05:00:01+00:00,05:00 - 05:10,2018,1,14
8,LsoLEjrDogU,2018-01-12,Bruno Mars - Finesse (Remix) [Feat. Cardi B] [...,Bruno Mars,10,2018-01-04T04:49:43.000Z,"Bruno Mars|""Finesse""|""Cardi B""|""Finesse Remix""...",57951412,1919583,73239,...,False,False,Finesse (Remix) Feat. Cardi B Available Now: h...,US,26.209503,2018-01-04 04:49:43+00:00,04:40 - 04:50,2018,1,12
9,xpVfcZ0ZcFM,2018-02-24,Drake - God’s Plan,DrakeVEVO,10,2018-02-17T05:00:01.000Z,"Drake new music|""Drake Gods Plan""|""Drake God’s...",47362934,2469057,31843,...,False,False,God’s Plan (Official Video)\n\nSong Available ...,CA,77.536051,2018-02-17 05:00:01+00:00,05:00 - 05:10,2018,2,24


# 14. Read all json files with the video categories.

In [ ]:
# already done at the beginning: the JSON files are stored
# inside the json_data dictionary

# some checks:
json_data.keys()

dict_keys(['CA', 'DE', 'FR', 'GB', 'IN', 'JP', 'KR', 'MX', 'RU', 'US'])

In [ ]:
print(json.dumps(json_data["US"], indent=2))
# convert the dictionary to JSON format for a more readable structure

# the dictionary has three main keys: kind, etag, and items.
# "items" contains the list of video categories, each identified by an id.

{
  "kind": "youtube#videoCategoryListResponse",
  "etag": "\"m2yskBQFythfE4irbTIeOgYYfBU/S730Ilt-Fi-emsQJvJAAShlR6hM\"",
  "items": [
    {
      "kind": "youtube#videoCategory",
      "etag": "\"m2yskBQFythfE4irbTIeOgYYfBU/Xy1mB4_yLrHy_BmKmPBggty2mZQ\"",
      "id": "1",
      "snippet": {
        "channelId": "UCBR8-60-B28hp2BmDPdntcQ",
        "title": "Film & Animation",
        "assignable": true
      }
    },
    {
      "kind": "youtube#videoCategory",
      "etag": "\"m2yskBQFythfE4irbTIeOgYYfBU/UZ1oLIIz2dxIhO45ZTFR3a3NyTA\"",
      "id": "2",
      "snippet": {
        "channelId": "UCBR8-60-B28hp2BmDPdntcQ",
        "title": "Autos & Vehicles",
        "assignable": true
      }
    },
    {
      "kind": "youtube#videoCategory",
      "etag": "\"m2yskBQFythfE4irbTIeOgYYfBU/nqRIq97-xe5XRZTxbknKFVe5Lmg\"",
      "id": "10",
      "snippet": {
        "channelId": "UCBR8-60-B28hp2BmDPdntcQ",
        "title": "Music",
        "assignable": true
      }
    },
    {
      "kind

# 15. For each country, determine how many videos have a category that is not assignable.

In [ ]:
# in df2, each video has a category_id.

# in the JSON files, inside "items" -> "snippet" -> "assignable",
# there is a boolean value (True or False).

# build a dataframe with categories per country

df_cat = pd.DataFrame([
    {
        "country": country,
        "category_id": item["id"],
        "assignable": item["snippet"]["assignable"]
    }
    for country, content in json_data.items()
    for item in content["items"]
])
# create df_cat with one row per category per country, including the "assignable" flag

# convert category_id to string in both dataframes before merging
# we use df_unique since we are considering individual videos
df_unique["category_id"] = df_unique["category_id"].astype(str)
df_cat["category_id"] = df_cat["category_id"].astype(str)

# merge videos with category information
df_merged = df_unique.merge(df_cat, on=["country", "category_id"], how="left")

# count videos with non-assignable categories per country
not_assignable = (
    df_merged[df_merged["assignable"] == False]
    .groupby("country")
    .size()
    .reset_index(name="num_not_assignable")
)

not_assignable

,country,num_not_assignable
0,CA,45
1,DE,32
2,FR,54
3,GB,1
4,IN,83
5,KR,74
6,MX,3
7,RU,154
8,US,4
